# 01 — Data Exploration
## NetGuard AI: ML-Powered Network Intrusion Detection

This notebook explores the NSL-KDD dataset to understand:
- Data structure and feature types
- Class distribution (normal vs attack)
- Attack type breakdown
- Feature correlations and distributions

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
print('Libraries loaded')

## 1. Load Dataset

In [ ]:
from netguard.preprocessing.loader import load_nsl_kdd

train_df = load_nsl_kdd('train')
test_df = load_nsl_kdd('test')

print(f'Training set: {train_df.shape}')
print(f'Test set: {test_df.shape}')
train_df.head()

## 2. Data Overview

In [ ]:
print('=== Data Types ===')
print(f'Numerical: {len(train_df.select_dtypes(include=["number"]).columns)}')
print(f'Categorical: {len(train_df.select_dtypes(include=["object"]).columns)}')
print()
print('=== Missing Values ===')
missing = train_df.isnull().sum()
print(f'Total missing: {missing.sum()}')
print()
print('=== Basic Statistics ===')
train_df.describe()

## 3. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Binary distribution
binary_counts = train_df['is_attack'].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0].pie(binary_counts, labels=['Normal', 'Attack'], autopct='%1.1f%%', colors=colors, startangle=90)
axes[0].set_title('Binary Classification Distribution')

# Attack categories
cat_counts = train_df['attack_cat'].value_counts()
cat_counts.plot(kind='bar', ax=axes[1], color='#3498db', edgecolor='black')
axes[1].set_title('Attack Category Distribution')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../docs/figures/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nAttack categories:')
print(cat_counts)

## 4. Feature Distributions

In [ ]:
# Top numeric features by variance
numeric_cols = train_df.select_dtypes(include=['number']).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ['is_attack']]

top_variance = train_df[numeric_cols].var().sort_values(ascending=False).head(8)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for idx, col in enumerate(top_variance.index):
    ax = axes[idx // 4][idx % 4]
    train_df[train_df['is_attack']==0][col].hist(ax=ax, alpha=0.5, label='Normal', color='#2ecc71', bins=30)
    train_df[train_df['is_attack']==1][col].hist(ax=ax, alpha=0.5, label='Attack', color='#e74c3c', bins=30)
    ax.set_title(col, fontsize=10)
    ax.legend(fontsize=7)

plt.suptitle('Feature Distributions: Normal vs Attack', fontsize=14)
plt.tight_layout()
plt.savefig('../docs/figures/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Correlation Matrix

In [ ]:
# Correlation with target
correlations = train_df[numeric_cols + ['is_attack']].corr()['is_attack'].drop('is_attack').sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
correlations.head(20).plot(kind='barh', ax=ax, color=['#e74c3c' if v > 0 else '#2ecc71' for v in correlations.head(20)])
ax.set_title('Top 20 Features Correlated with Attack Label')
ax.set_xlabel('Correlation')
plt.tight_layout()
plt.savefig('../docs/figures/feature_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 10 correlated features:')
print(correlations.head(10))

## 6. Key Findings

Summary of data exploration:
- **Class balance**: The dataset has X% normal and Y% attack traffic
- **Attack types**: DoS is the most common, followed by Probe
- **Key features**: `src_bytes`, `dst_bytes`, `count`, `serror_rate` are most discriminative
- **Missing values**: None — dataset is clean
- **Next step**: Feature engineering and model training (see notebook 02)